# EDEF Tree Classification Quickstart

Tree classification uses raw margins/logits, not probability-output trees.

For binary classification, EDEF decomposes realized binary log-loss improvement. For multiclass classification, EDEF decomposes realized softmax log-loss improvement.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier

import edef


In [ ]:
rng = np.random.default_rng(123)

X = rng.normal(size=(500, 4))

score = (
    1.2 * X[:, 0]
    - 0.9 * X[:, 1]
    + 0.6 * X[:, 2] * X[:, 3]
)

y = (score > np.median(score)).astype(float)

feature_names = [
    "x1_linear",
    "x2_linear",
    "x3_interact",
    "x4_interact",
]


In [ ]:
model = GradientBoostingClassifier(
    n_estimators=30,
    max_depth=3,
    learning_rate=0.07,
    random_state=0,
)

model.fit(X, y)


In [ ]:
explainer = edef.TreeExplainer(
    model,
    baseline=X.mean(axis=0),
    loss="log_loss",
    target=1,
    feature_names=feature_names,
)

result = explainer(X, y)


In [ ]:
print(result)


In [ ]:
result.to_frame()


In [ ]:
ax = result.plot()
ax.set_title("Tree EDEF contributions to realized binary log-loss fit")
plt.show()


In [ ]:
grouped = result.group(
    ["main", "main", "interaction", "interaction"]
)

print(grouped)


In [ ]:
ax = grouped.plot()
ax.set_title("Grouped tree classification EDEF contributions")
plt.show()


The binary tree example uses a GradientBoostingClassifier because it has additive raw-margin outputs.

TreeIG supplies exact margin-jump traces, and EDEF converts those margin jumps into exact finite log-loss improvements.


## Multiclass Tree Classification


In [ ]:
n_obs = 600
n_features = 4
n_classes = 3

X_multi = rng.normal(size=(n_obs, n_features))

scores = np.column_stack(
    [
        1.2 * X_multi[:, 0] - 0.3 * X_multi[:, 1],
        1.0 * X_multi[:, 1] + 0.4 * X_multi[:, 2],
        -0.8 * X_multi[:, 0] + 0.6 * X_multi[:, 3],
    ]
)

y_multi = np.argmax(
    scores + rng.normal(scale=0.75, size=(n_obs, n_classes)),
    axis=1,
)


In [ ]:
multi_model = GradientBoostingClassifier(
    n_estimators=25,
    max_depth=3,
    learning_rate=0.07,
    random_state=0,
)

multi_model.fit(X_multi, y_multi)


In [ ]:
multi_explainer = edef.TreeExplainer(
    multi_model,
    baseline=X_multi.mean(axis=0),
    loss="multiclass_log_loss",
    feature_names=["x1", "x2", "x3", "x4"],
)

multi_result = multi_explainer(X_multi, y_multi)


In [ ]:
print(multi_result)


In [ ]:
ax = multi_result.plot()
ax.set_title("Tree EDEF contributions to realized multiclass log-loss fit")
plt.show()


For multiclass tree models, EDEF calls TreeIG once per class margin, merges all crossing events in path order, and computes exact softmax log-loss improvements from the full margin vector.
